In [ ]:
import mlflow

# ============================================================
# 1. Connexion au serveur MLflow
# ============================================================

# mlflow.set_tracking_uri("http://mlflow:5000")   # Pour utilisation dans un conteneur Docker

mlflow.set_tracking_uri("http://localhost:5000")   # Pour utilisation en local

# ============================================================
# 2. Récupérer l'expérience
# ============================================================
experiment_name = "rakuten_xgb_fusion"
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"L'expérience '{experiment_name}' n'existe pas.")

# ============================================================
# 3. Sélectionner le meilleur run (trié par accuracy DESC)
# ============================================================
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)

if len(runs) == 0:
    raise ValueError("Aucun run trouvé dans MLflow pour cette expérience.")

best_run = runs[0]
best_run_id = best_run.info.run_id
best_acc = best_run.data.metrics["accuracy"]

print("📌 Best run ID :", best_run_id)
print("📈 Best accuracy :", best_acc)

# ============================================================
# 4. Charger le meilleur modèle XGBoost
# ============================================================
model_uri = f"runs:/{best_run_id}/xgb_model"
model = mlflow.xgboost.load_model(model_uri)

print("✅ Modèle chargé avec succès :", model_uri)



CI-DESSOUS CODE A INTEGRER DANS PREDICT.PY POUR CHARGER LE MODELE

In [ ]:
import mlflow  # bibliothèque MLflow

mlflow.set_tracking_uri("http://mlflow:5000")  # connexion au serveur MLflow dans Docker

client = mlflow.tracking.MlflowClient()  # client pour interagir avec MLflow
exp = client.get_experiment_by_name("rakuten_xgb_fusion")  # récupère l'expérience

best_run = client.search_runs(  # sélectionne le meilleur run
    [exp.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)[0]

model = mlflow.xgboost.load_model(  # charge le modèle XGBoost depuis MLflow
    f"runs:/{best_run.info.run_id}/xgb_model"
)

print("Loaded model from run:", best_run.info.run_id)  # confirmation
